# Colab notebook for fine-tuning

In [1]:
%pip install transformers datasets pandas scikit-learn sentencepiece torch accelerate -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import logging
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import torch

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
LOGGER = logging.getLogger(__name__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
LOGGER.info(f"Using device: {device}")

MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
TASK_PREFIX = "translate slang to standard: "

c:\Users\ganim\Documents\school\3.2\NLP\Finals\chronically-online-translator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: Using device: cpu


In [3]:
def load_data(train_path, test_path):
    """Load train/test CSVs into Hugging Face Datasets."""
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)
    
    LOGGER.info(f"Loaded {len(train_dataset)} training examples")
    LOGGER.info(f"Loaded {len(test_dataset)} test examples")
    
    return train_dataset, test_dataset

def preprocess_function(examples, tokenizer):
    """Tokenize inputs with task prefix and targets as labels."""
    inputs = [TASK_PREFIX + ex for ex in examples["source_text"]]
    targets = examples["target_text"]
    
    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )
    
    # Tokenize targets as labels
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )
    
    labels["input_ids"] = [
    [(token if token != tokenizer.pad_token_id else -100) for token in label]
    for label in labels["input_ids"]
]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [4]:
train_dataset, test_dataset = load_data("../data/processed/train.csv", "../data/processed/test.csv")

INFO: Loaded 854 training examples
INFO: Loaded 151 test examples


In [5]:
LOGGER.info(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
LOGGER.info(f"Model loaded. Parameters: {model.num_parameters():,.0f}")

INFO: Loading google/flan-t5-small...
INFO: HTTP Request: HEAD https://huggingface.co/google/flan-t5-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-small/0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab/config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/google/flan-t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-small/0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab/tokenizer_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-small/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/goog

In [6]:
LOGGER.info("Tokenizing training dataset...")
train_dataset = train_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=["source_text", "target_text"],
)

LOGGER.info("Tokenizing test dataset...")
test_dataset = test_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=["source_text", "target_text"],
)

LOGGER.info(f"Train dataset columns: {train_dataset.column_names}")
LOGGER.info("Tokenization complete!")

INFO: Tokenizing training dataset...
Map: 100%|██████████| 854/854 [00:00<00:00, 3138.63 examples/s]
INFO: Tokenizing test dataset...
Map: 100%|██████████| 151/151 [00:00<00:00, 3144.56 examples/s]
INFO: Train dataset columns: ['input_ids', 'attention_mask', 'labels']
INFO: Tokenization complete!


In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir="backend/fine_tuned_model",
    num_train_epochs=5,  # Adjust to 5-10 as needed
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    save_total_limit=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    predict_with_generate=True,
    learning_rate=1e-4,
    seed=42,
    fp16=device == "cuda",  # Use fp16 if GPU available
)

LOGGER.info("Training configuration ready")

INFO: Training configuration ready


In [8]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

LOGGER.info("Starting training...")
trainer.train()

INFO: Starting training...
c:\Users\ganim\Documents\school\3.2\NLP\Finals\chronically-online-translator\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,1.181166
2,1.545654,0.764044
3,1.545654,0.563698
4,0.833102,0.488142
5,0.833102,0.450320


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.36it/s]
c:\Users\ganim\Documents\school\3.2\NLP\Finals\chronically-online-translator\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.08it/s]
c:\Users\ganim\Documents\school\3.2\NLP\Finals\chronically-online-translator\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.25it/s]
c:\Users\ganim\Documents\school\3.2\NLP\Finals\chronically-online-translator\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device 

TrainOutput(global_step=270, training_loss=1.044939973619249, metrics={'train_runtime': 1669.4087, 'train_samples_per_second': 2.558, 'train_steps_per_second': 0.162, 'total_flos': 198438113771520.0, 'train_loss': 1.044939973619249, 'epoch': 5.0})

In [16]:
output_dir = "../backend/fine_tuned_model"
import shutil

# Clear existing output directory to avoid file locking issues
if Path(output_dir).exists():
    shutil.rmtree(output_dir)

LOGGER.info(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=False)
LOGGER.info("Model and tokenizer saved!")
LOGGER.info("Training complete!")

INFO: Saving model to ../backend/fine_tuned_model...
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.72it/s]
INFO: Model and tokenizer saved!
INFO: Training complete!


In [10]:
# Uncomment to share weights early via Google Drive
# from google.colab import files
# import shutil
# 
# # Zip the model checkpoint
# model_checkpoint = "backend/fine_tuned_model/checkpoint-50"  # Adjust checkpoint number
# shutil.make_archive("checkpoint", "zip", "backend/fine_tuned_model", "checkpoint-50")
# 
# # Download to local, then upload manually to Google Drive
# files.download("checkpoint.zip")
# LOGGER.info("Checkpoint zipped and ready for download to Google Drive")

In [17]:
# Load the trained model for inference
trained_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
trained_tokenizer = AutoTokenizer.from_pretrained(output_dir)

# Test example
test_input = "translate slang to standard: yo that's lowkey fire bro"
inputs = trained_tokenizer.encode(test_input, return_tensors="pt")
outputs = trained_model.generate(inputs, max_length=128)
result = trained_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Input: {test_input}")
print(f"Output: {result}")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 7908.60it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Input: translate slang to standard: yo that's lowkey fire bro
Output: hey, i think i'm just really tired.
